# Compression

When working with large data files compression might be worth considering. This demo will describe compression options in ASDF.

In [1]:
#!pip install asdf

Let's start by making a simple ASDF file. With a 2k x 2k array and some metadata

In [2]:
import asdf
import numpy as np
af = asdf.AsdfFile()
af['arr'] = np.zeros((2048, 2048), dtype='u4')
af['meta'] = {"quality": "excellent"}
af.write_to("example.asdf")

This example file is ~16 MB (almost entirely the 2k x 2k array).

In [3]:
import os
os.path.getsize("example.asdf")

16778012

Since the array is all zeros stored as unsigned ints it is highly compressible. Asdf provides a few options for compressing arrays.

In all cases the YAML header will remain uncompressed and all compression options will only impact the binary blocks (often used for array data).

# All Array Compression
Providing the all_array_compression argument to [AsdfFile.write_to](https://www.asdf-format.org/projects/asdf/en/latest/api/asdf.AsdfFile.html#asdf.AsdfFile.write_to) will cause all arrays to be compressed using the provided algorithm.

In [4]:
af.write_to("bzp2_compressed.asdf", all_array_compression="bzp2")
os.path.getsize("bzp2_compressed.asdf")

841

This file can be opened as usual and asdf will transparently decompress the data when accessed.

In [5]:
af = asdf.open("bzp2_compressed.asdf")
print(af["arr"].max())

0


# Using AsdfConfig
Instead of providing `all_array_compression` to [AsdfFile.write_to](https://www.asdf-format.org/projects/asdf/en/latest/api/asdf.AsdfFile.html#asdf.AsdfFile.write_to), the default compression algorithm (`None` meaning don't compress the array) can be changed by modifying the current [AsdfConfig](https://www.asdf-format.org/projects/asdf/en/latest/api/asdf.config.AsdfConfig.html#asdf.config.AsdfConfig).

In [6]:
with asdf.config_context() as cfg:
    cfg.all_array_compression = "zlib"
    af = asdf.AsdfFile()
    af["arr"] = np.ones((2048, 2048), dtype="i4")
    af.write_to("zlib_compressed.asdf")

In [7]:
af = asdf.open("zlib_compressed.asdf")
af.get_array_compression(af["arr"])

'zlib'

# AsdfFile.set_array_compression
The options we've covered so far use the same compression algorithm for all arrays. This might be sub-optimal for some file structures and applications. For more fine-grained control, [AsdfFile.set_array_compression](https://www.asdf-format.org/projects/asdf/en/latest/api/asdf.AsdfFile.html#asdf.AsdfFile.set_array_compression) can be used to set the compression algorithm on a per-array basis.

In [8]:
af = asdf.AsdfFile()
af["uint_array"] = np.zeros((2048, 2048), dtype="u4")
af["double_array"] = np.random.random(10000)
af.set_array_compression(af["uint_array"], "bzp2")
# the double array won't compress well so we don't compress it
af.write_to("mixed_compression.asdf")

# AsdfFile.get_array_compression
Asdf will track what compression was used for an array and reuse the same compression algorithm if the file is rewritten. [AsdfFile.get_array_compression](https://www.asdf-format.org/projects/asdf/en/latest/api/asdf.AsdfFile.html#asdf.AsdfFile.get_array_compression) can be used to check what algorithm asdf will use to compress a given array.

In [9]:
af = asdf.open("mixed_compression.asdf")
af.get_array_compression(af["uint_array"])

'bzp2'

In [10]:
af.get_array_compression(af["double_array"]) is None

True

# Compression algorithms
So far we've used the `bzp2` and `zlib` algorithms. Support for these algorithms is required by the [ASDF-standard](https://www.asdf-format.org/projects/asdf-standard/en/1.1.1/file_layout.html#compression). The [asdf compression extension API](https://www.asdf-format.org/projects/asdf/en/latest/asdf/extending/compressors.html#binary-block-compressors) can be used to add new compression algorithms. Please see the linked documentation for more details.